In [1]:
import numpy as np
import h5py
from scipy.fft import fft, fftshift, ifft, ifftshift


In [2]:
def norm_check(psi):

    err = np.abs(np.vdot(psi, psi) -1.)
    if err > 1e-14:
        raise ValueError(rf'Normalization Problems: err = {err}')


def U_k(theta_list, k_val):

    return np.exp(1.j * 0.5 * k_val * (theta_list - np.pi)**2 )

def U_T(n_list, T_val):

    return np.exp(-1.j * 0.5 * T_val * (n_list)**2 )

def rappr_ang_to_n(psi_ang):

    return fftshift(fft(psi_ang, norm='ortho'))

def rappr_n_to_ang(psi_n):

    return ifft(ifftshift(psi_n), norm='ortho')

def qswm_pass(psi, Opp_Uk, Opp_UT):

    psi = Opp_Uk * psi


    psi = rappr_ang_to_n(psi)


    psi = Opp_UT * psi

    psi =  rappr_n_to_ang(psi)

    return psi


In [ ]:
nq = 10
dim_H = 2 ** nq
n_iterations = 5000

ifsave = False


k_val = np.sqrt(3) 

T_val = np.sqrt(2/3)

loc_len = np.square(k_val * np.pi) / 3

eta = loc_len / dim_H  

print(rf'nq:{nq:n}, dim_H:{dim_H:n}, k:{k_val:.2e}, T:{T_val:.2e}, eta:{eta:.2e}')

n_eigvl = np.arange(dim_H) - dim_H // 2
ang_eigvl =  2 * np.pi * np.arange(dim_H) / dim_H

# print(r'n:', n_eigvl)
# print(r'theta:', ang_eigvl)

n0_list = [dim_H //2]
n0_list = np.arange(dim_H)

psi_n0 = np.zeros(shape=dim_H, dtype=np.complex128)
psi_n0[n0_list] = 1.
psi_n0 = psi_n0 / np.linalg.norm(psi_n0)

Opp_Uk = U_k(ang_eigvl, k_val)
Opp_UT = U_T(n_eigvl, T_val)

# print(psi_n0)


nq:10, dim_H:1024, k:1.73e+00, T:8.16e-01, eta:9.64e-03


In [ ]:

with h5py.File(f'Data/qsm_nq{nq:n}_k{k_val:.2f}_T{T_val:.2f}.h5', 'w') as f:

    if ifsave is True:
        grp = f.create_dataset('psi', shape = (1, dim_H), maxshape = (None, dim_H), 
            dtype = 'complex128', chunks = (1, dim_H), compression='lzf')

        grp.attrs['nq'] = nq
        grp.attrs['k'] = k_val
        grp.attrs['T'] = T_val

        grp[0] = psi_n0

    psi_now = rappr_n_to_ang(psi_n0)

    for t in range(n_iterations):

        psi_next = qswm_pass(psi_now, Opp_Uk, Opp_UT)

        psi_now = psi_next

        psi_next = rappr_ang_to_n(psi_next) 
        

        if ifsave is True: 
            
            grp.resize(grp.shape[0] + 1, axis = 0)

            grp[-1] = psi_next

        if (t % 1000 == 0):
            psi_now = psi_now / np.sqrt(np.vdot(psi_now, psi_now).real)